Problem Statement: 
You are a Data Engineer on the Growth Analytics team at Meta. Your task is to sessionize raw clickstream data from
the mobile app. A session is defined as a continuous sequence of events from a single user where the gap between any
two consecutive events is less than 30 minutes. If the gap between two events exceeds 30 minutes, a new session
begins.
Given a table of raw user events (user ID, event timestamp, event type), produce an output table where every event
row is annotated with a monotonically increasing session_id scoped per user. The final output should also include
the session's start time, end time, and total event count per session.
Business motivation: Session-level engagement metrics (average session length, events per session, drop-off within
sessions) are foundational to ranking and recommendation system feature engineering.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window

data = [
    ("u001", "2024-01-15 09:00:00", "page_view"),
("u001", "2024-01-15 09:12:00", "click"),
("u001", "2024-01-15 09:45:00", "page_view"),
("u001", "2024-01-15 09:58:00", "purchase"),
("u002", "2024-01-15 10:00:00", "page_view"),
]

schema = ["user_id", "event_ts", "event_type"]
raw_events = spark.createDataFrame(data, schema).withColumn("event_ts", to_timestamp("event_ts"))
raw_events.display()

In [0]:
# Define single reusable window spec
user_time_window = (
    Window.partitionBy("user_id")
    .orderBy("event_ts")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# a separate unbounded window is neededfor lag (no frame restriction)
lag_window = Window.partitionBy("user_id").orderBy("event_ts")

# Detect session boundaries
events_with_lag = raw_events.withColumn(
    "prev_ts", lag("event_ts",1).over(lag_window)) \
    .withColumn("gap_seconds", when(col("prev_ts").isNull() | (unix_timestamp("event_ts") - unix_timestamp("prev_ts") >= 1800), 1).otherwise(0))
events_with_lag.display()

In [0]:
# Cumulative sum over the new_session flag -> session counter
events_with_session_num = events_with_lag.withColumn("session_num", sum("gap_seconds").over(user_time_window))
events_with_session_num.display()

In [0]:
# construct a globally unique session_id
events_with_session_id = events_with_session_num.withColumn("session_id", concat_ws("_", col("user_id"), col("session_num").cast("string")))
events_with_session_id.display()

In [0]:
# compute session-leve aggregates and join back
session_agg = (
    events_with_session_id.groupBy("user_id", "session_id").agg(min("event_ts").alias("session_start"), max("event_ts").alias("session_end"), count("*").alias("session_event_count"))
)
session_agg.display()

In [0]:
# broadcast the smaller session aggregates to avoid a shuffle on the event side

final_df = events_with_session_id.join(session_agg, on=["user_id","session_id"], how = "left").select("user_id", "session_id", "event_ts", "event_type", "session_start", "session_end", "session_event_count").orderBy("user_id", "event_ts")
final_df.display()

Databricks visualization. Run in Databricks to view.